In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

tqdm.pandas()

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("simhadrisadaram/mimic-cxr-dataset")

print("Path to dataset files:", path)

In [ ]:
os.listdir(path)

In [ ]:
path = Path(path)
TRAIN_CSV  = path / "mimic_cxr_aug_validate.csv"
VAL_CSV    = path / "mimic_cxr_aug_train.csv"
IMAGE_ROOT = path / "official_data_iccv_final"

assert TRAIN_CSV.exists()
assert VAL_CSV.exists()
assert IMAGE_ROOT.exists()

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)

print("Train:", train_df.shape)
print("Val:", val_df.shape)

In [ ]:
print(train_df.columns.tolist())

In [ ]:
DROP_COLS = ["Unnamed: 0", "Unnamed: 0.1"]

train_df = train_df.drop(columns=DROP_COLS, errors="ignore")
val_df   = val_df.drop(columns=DROP_COLS, errors="ignore")

In [ ]:
train_df = train_df.rename(columns={
    "subject_id": "patient_id",
    "image": "image_path",
    "view": "ViewPosition",
    "text": "findings_gt",
    "text_augment": "findings_noisy"
})

val_df = val_df.rename(columns={
    "subject_id": "patient_id",
    "image": "image_path",
    "view": "ViewPosition",
    "text": "findings_gt",
    "text_augment": "findings_noisy"
})

In [ ]:
def resolve_image_paths(rel_paths):
    abs_paths = []
    for p_str in rel_paths:
        if not p_str:
            continue
        p = Path(p_str)
        if not p.parts:
            continue
        if p.parts[0] == "files":
            p = Path(*p.parts[1:])
        abs_paths.append(str(IMAGE_ROOT / p))
    return abs_paths

In [ ]:
train_df["image_path"] = train_df["image_path"].progress_apply(resolve_image_paths)
val_df["image_path"]   = val_df["image_path"].progress_apply(resolve_image_paths)

In [ ]:
VALID_VIEWS = ["PA", "AP"]

train_df = train_df[train_df["ViewPosition"].isin(VALID_VIEWS)]
val_df   = val_df[val_df["ViewPosition"].isin(VALID_VIEWS)]

print("Train after view filter:", train_df.shape)
print("Val after view filter:", val_df.shape)

In [ ]:
# correct paths
TRAIN_CSV = path / "mimic_cxr_aug_train.csv"
VAL_CSV   = path / "mimic_cxr_aug_validate.csv"
IMAGE_ROOT = path / "official_data_iccv_final" / "files"

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)

# drop junk
train_df = train_df.drop(columns=["Unnamed: 0", "Unnamed: 0.1"], errors="ignore")
val_df   = val_df.drop(columns=["Unnamed: 0", "Unnamed: 0.1"], errors="ignore")

# rename
train_df = train_df.rename(columns={
    "subject_id": "patient_id",
    "image": "image_path",
    "text": "findings_gt",
    "text_augment": "findings_noisy"
})
val_df = val_df.rename(columns={
    "subject_id": "patient_id",
    "image": "image_path",
    "text": "findings_gt",
    "text_augment": "findings_noisy"
})

# resolve paths safely
def resolve_image_paths(rel_paths):
    if isinstance(rel_paths, str):
        rel_paths = [rel_paths]

    abs_paths = []
    for p_str in rel_paths:
        p = Path(p_str)
        if p.parts[0] == "files":
            p = Path(*p.parts[1:])
        abs_paths.append(str(IMAGE_ROOT / p))
    return abs_paths

train_df["image_path"] = train_df["image_path"].apply(resolve_image_paths)
val_df["image_path"]   = val_df["image_path"].apply(resolve_image_paths)

# no view filtering
train_df["ViewPosition"] = "UNKNOWN"
val_df["ViewPosition"] = "UNKNOWN"


In [ ]:
print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)

print("\nTrain columns:")
print(train_df.columns.tolist())

In [ ]:
sample = train_df.iloc[0]["image_path"]

print("image_path type:", type(sample))
print("number of images:", len(sample))
print("first image path:", sample[0])

In [ ]:
print("First image exists:", os.path.exists(sample[0]))

In [ ]:
import random

idx = random.randint(0, len(train_df) - 1)
paths = train_df.iloc[idx]["image_path"]

print("Random sample index:", idx)
for p in paths:
    print(p, "→", os.path.exists(p))

In [ ]:
import os
import ast
import pandas as pd
from pathlib import Path

DATASET_ROOT = Path("/kaggle/input/mimic-cxr-dataset")
BASE_DIR = DATASET_ROOT / "official_data_iccv_final"
IMAGE_ROOT = BASE_DIR / "files"

TRAIN_CSV = "/kaggle/input/mimic-cxr-dataset/mimic_cxr_aug_train.csv"       # small (500)
VAL_CSV   = "/kaggle/input/mimic-cxr-dataset/mimic_cxr_aug_validate.csv"    # large (64k)

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)

DROP_COLS = ["Unnamed: 0", "Unnamed: 0.1"]
train_df = train_df.drop(columns=DROP_COLS, errors="ignore")
val_df   = val_df.drop(columns=DROP_COLS, errors="ignore")

train_df = train_df.rename(columns={
    "subject_id": "patient_id",
    "image": "image_path",
    "text": "findings_gt",
    "text_augment": "findings_noisy"
})

val_df = val_df.rename(columns={
    "subject_id": "patient_id",
    "image": "image_path",
    "text": "findings_gt",
    "text_augment": "findings_noisy"
})

def resolve_image_paths(rel_paths):
    # If stringified Python list → parse it
    if isinstance(rel_paths, str):
        try:
            rel_paths = ast.literal_eval(rel_paths)
        except Exception:
            rel_paths = [rel_paths]

    abs_paths = []
    for p_str in rel_paths:
        p = Path(p_str)
        if p.parts[0] == "files":
            p = Path(*p.parts[1:])   # strip 'files'
        abs_paths.append(str(IMAGE_ROOT / p))

    return abs_paths

train_df["image_path"] = train_df["image_path"].apply(resolve_image_paths)
val_df["image_path"]   = val_df["image_path"].apply(resolve_image_paths)

train_df["ViewPosition"] = "UNKNOWN"
val_df["ViewPosition"]   = "UNKNOWN"

def clean_text(x):
    if x is None or pd.isna(x):
        return None
    x = str(x).strip()
    return x if len(x) > 20 else None

train_df["findings_gt"] = train_df["findings_gt"].apply(clean_text)
train_df["findings_noisy"] = train_df["findings_noisy"].apply(clean_text)

val_df["findings_gt"] = val_df["findings_gt"].apply(clean_text)
val_df["findings_noisy"] = val_df["findings_noisy"].apply(clean_text)

train_df = train_df.dropna(subset=["findings_gt"])
val_df   = val_df.dropna(subset=["findings_gt"])

train_df["gt_score"] = 1.0
train_df["noisy_score"] = train_df["findings_noisy"].apply(
    lambda x: 0.0 if x is not None else 1.0
)

val_df["gt_score"] = 1.0
val_df["noisy_score"] = val_df["findings_noisy"].apply(
    lambda x: 0.0 if x is not None else 1.0
)

assert train_df["image_path"].apply(lambda x: isinstance(x, list)).all()
assert train_df["image_path"].apply(lambda x: len(x) > 0).all()
assert train_df["image_path"].apply(lambda x: all(os.path.exists(p) for p in x)).all()
assert train_df["findings_gt"].notna().all()

print("DATASET FIXED AND VERIFIED")
print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)

In [ ]:
!pip install -q torch torchvision transformers pillow einops

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import os

In [ ]:
def get_image_transform(image_size=224):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

In [ ]:
class MIMICCXRDataset(Dataset):
    def __init__(
        self,
        csv_path,
        tokenizer,
        image_size=224,
        max_text_len=512,
        use_noisy=False
    ):
        self.df = pd.read_csv(csv_path)
        self.tokenizer = tokenizer
        self.transform = get_image_transform(image_size)
        self.max_text_len = max_text_len
        self.use_noisy = use_noisy

    def __len__(self):
        return len(self.df)

    def load_images(self, image_paths):
        images = []
        for p in image_paths:
            img = Image.open(p).convert("RGB")
            img = self.transform(img)
            images.append(img)
        return images

    def tokenize_text(self, text):
        return self.tokenizer(
            text,
            padding=False,
            truncation=True,
            max_length=self.max_text_len,
            return_tensors="pt"
        )

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_paths = eval(row["image_path"]) if isinstance(row["image_path"], str) else row["image_path"]
        images = self.load_images(image_paths)

        # Choose which text to train on
        if self.use_noisy and pd.notna(row["findings_noisy"]):
            text = row["findings_noisy"]
            score = row["noisy_score"]
        else:
            text = row["findings_gt"]
            score = row["gt_score"]

        tokenized = self.tokenize_text(text)

        return {
            "images": images,                   # List[Tensor]
            "input_ids": tokenized["input_ids"].squeeze(0),
            "attention_mask": tokenized["attention_mask"].squeeze(0),
            "score": torch.tensor(score, dtype=torch.float32)
        }

In [ ]:
def mimic_cxr_collate(batch):
    # Images: list of list of tensors
    batch_images = [item["images"] for item in batch]

    # Text
    input_ids = torch.nn.utils.rnn.pad_sequence(
        [item["input_ids"] for item in batch],
        batch_first=True,
        padding_value=0
    )

    attention_mask = torch.nn.utils.rnn.pad_sequence(
        [item["attention_mask"] for item in batch],
        batch_first=True,
        padding_value=0
    )

    scores = torch.stack([item["score"] for item in batch])

    return {
        "images": batch_images,   # still list → handled by model
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "scores": scores
    }

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

In [ ]:
train_dataset = MIMICCXRDataset(
    csv_path=TRAIN_CSV,
    tokenizer=tokenizer,
    image_size=224,
    use_noisy=False
)

val_dataset = MIMICCXRDataset(
    csv_path=VAL_CSV,
    tokenizer=tokenizer,
    image_size=224,
    use_noisy=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=mimic_cxr_collate,
    num_workers=2
)

batch = next(iter(train_loader))
print("Batch keys:", batch.keys())
print("Images per sample:", len(batch["images"][0]))
print("Text shape:", batch["input_ids"].shape)
print("Scores:", batch["scores"])